<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/rosg_test6b_ifs_global_pcf_deltatau_inflection_dsi_audit_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test6b — DeltaTau Inflection Global IFS/PCF Stochastic Overlap DSI Audit

This notebook starts from the Test5 global IFS/PCF contact guardrail, enables endogenous stochastic overlap oscillations, and restricts the DSI audit to the local DeltaTau sigmoid inflection window.

Reference DeltaTau parameters: `Zc_i = 1.05 + (i-1) ln(q)`, `width_Z = 0.34`, audit window `[Zc_i - 2w_Z, Zc_i + 2w_Z]`.

In [ ]:

# ROSG_Test6b_IFS_Global_PCF_DeltaTau_Inflection_DSI_Audit_CORRECTED.py
# Corrected Test6b DeltaTau-inflection audit: endogenous stochastic overlap oscillations on the global IFS/PCF contact set.
# Starting point: Test5 global IFS/PCF contact guardrail.
# Core correction: stochastic overlap oscillations are enabled in the real branch;
# only an external, hand-tuned log-periodic DSI template is forbidden.

from __future__ import annotations

import itertools
import json
import math
import os
import hashlib
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from fractions import Fraction
from typing import Dict, List, Tuple, Set, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.linalg import eigvalsh
except Exception:
    eigvalsh = None

Point = Tuple[Fraction, ...]
Word = Tuple[int, ...]
Edge = Tuple[Word, Word]


@dataclass(frozen=True)
class IFSModel:
    name: str
    dim: int
    q: int
    translations: Tuple[Tuple[int, ...], ...]
    boundary_vertices: Tuple[Tuple[int, ...], ...]
    expected_pcf: bool = True
    note: str = ""

    @property
    def r(self) -> float:
        return 1.0 / float(self.q)

    @property
    def expected_Z_period(self) -> float:
        # Geometric contraction period in the operational variable Z=ln(DeltaTau/t_P).
        return math.log(float(self.q))


def frac_point(coords: Tuple[int, ...]) -> Point:
    return tuple(Fraction(c, 1) for c in coords)


def apply_map_to_point(point: Point, trans: Tuple[int, ...], q: int) -> Point:
    return tuple((point[k] + Fraction(trans[k], 1)) / q for k in range(len(point)))


def apply_word_to_point(point: Point, word: Word, model: IFSModel) -> Point:
    p = point
    for symbol in word:
        p = apply_map_to_point(p, model.translations[symbol], model.q)
    return p


def enumerate_words(n_symbols: int, level: int) -> List[Word]:
    return list(itertools.product(range(n_symbols), repeat=level))


def cell_vertices(model: IFSModel, word: Word) -> Tuple[Point, ...]:
    return tuple(apply_word_to_point(frac_point(v), word, model) for v in model.boundary_vertices)


def canonical_edge(a: Word, b: Word) -> Edge:
    return (a, b) if a <= b else (b, a)


def build_global_contact_structure(model: IFSModel, level: int) -> Dict[str, Any]:
    """Test5 guardrail: exact same-level global IFS/PCF contact set.

    Nodes are same-level IFS cells. Authorized overlap edges are exactly pairs of
    same-level cells sharing a rational PCF boundary/contact point. This excludes
    local fiber edges and inter-level shortcuts from the real branch.
    """
    words = enumerate_words(len(model.translations), level)
    point_to_cells: Dict[Point, Set[Word]] = {}
    for w in words:
        for p in cell_vertices(model, w):
            point_to_cells.setdefault(p, set()).add(w)
    contact_points = {p: cells for p, cells in point_to_cells.items() if len(cells) >= 2}
    edge_to_points: Dict[Edge, Set[Point]] = {}
    for p, cells in contact_points.items():
        for a, b in itertools.combinations(sorted(cells), 2):
            edge = canonical_edge(a, b)
            edge_to_points.setdefault(edge, set()).add(p)
    return {
        "model": model,
        "level": level,
        "words": words,
        "point_to_cells": point_to_cells,
        "contact_points": contact_points,
        "edge_to_points": edge_to_points,
        "max_contact_incidence": max((len(c) for c in contact_points.values()), default=0),
        "min_contact_incidence": min((len(c) for c in contact_points.values()), default=0),
    }


def guardrail_report(structure: Dict[str, Any], extra_edges: Set[Edge] | None = None) -> Dict[str, Any]:
    model = structure["model"]
    level = structure["level"]
    contact_edges = set(structure["edge_to_points"].keys())
    tested_edges = set(contact_edges)
    if extra_edges:
        tested_edges |= extra_edges
    out_of_contact = tested_edges - contact_edges
    same_level_pass = all(len(a) == level and len(b) == level for a, b in tested_edges)
    contact_only_pass = len(out_of_contact) == 0
    finite_contact_set_pass = len(structure["contact_points"]) < math.inf
    pcf_declared_pass = bool(model.expected_pcf)
    return {
        "model": model.name,
        "level": level,
        "n_cells": len(structure["words"]),
        "n_contact_points": len(structure["contact_points"]),
        "n_contact_edges": len(contact_edges),
        "max_contact_incidence": structure["max_contact_incidence"],
        "min_contact_incidence": structure["min_contact_incidence"],
        "same_level_pass": same_level_pass,
        "contact_only_pass": contact_only_pass,
        "finite_contact_set_pass": finite_contact_set_pass,
        "pcf_declared_pass": pcf_declared_pass,
        "global_ifs_pcf_guardrail_pass": bool(same_level_pass and contact_only_pass and finite_contact_set_pass and pcf_declared_pass),
        "n_out_of_contact_edges": len(out_of_contact),
    }


def build_invalid_local_fiber_edges(words: List[Word], contact_edges: Set[Edge], cap: int = 64) -> Set[Edge]:
    bad: Set[Edge] = set()
    for a, b in itertools.combinations(sorted(words), 2):
        e = canonical_edge(a, b)
        if e not in contact_edges:
            bad.add(e)
        if len(bad) >= max(1, min(cap, len(words) // 2)):
            break
    return bad


def sigmoid(x, center: float, width: float):
    return 1.0 / (1.0 + np.exp(-(np.asarray(x) - center) / width))

# DeltaTau sigmoid inflection parameters recovered from
# ROSG_Test_H1_DeltaTau_Resolution_Coupling_report.json
DELTATAU_ZC0 = 1.05
DELTATAU_WIDTH_Z = 0.34
DELTATAU_WINDOW_K = 2.0

def delta_tau_inflection_center(model: IFSModel, level: int) -> float:
    """Generalized DeltaTau inflection center.

    Reference notebook uses Zc_i = Zc0 + (i-1)*ln(2).
    For multi-IFS tests we generalize ln(2) to ln(q), so each PCF model
    is scanned in its own contraction-locked DeltaTau inflection window.
    """
    return float(DELTATAU_ZC0 + (level - 1) * math.log(float(model.q)))

def delta_tau_inflection_window(model: IFSModel, level: int) -> Tuple[float, float, float]:
    center = delta_tau_inflection_center(model, level)
    return (center - DELTATAU_WINDOW_K * DELTATAU_WIDTH_Z,
            center + DELTATAU_WINDOW_K * DELTATAU_WIDTH_Z,
            center)


def edge_base_weight(structure: Dict[str, Any], edge: Edge) -> float:
    model: IFSModel = structure["model"]
    level: int = structure["level"]
    multiplicity = len(structure["edge_to_points"].get(edge, []))
    rg_scale = float(model.q ** (-0.15 * level))
    return rg_scale * max(1, multiplicity)


def make_ou_process(n_steps: int, n_edges: int, rng: np.random.Generator, dz: float, corr_Z: float = 0.55) -> np.ndarray:
    """Generate endogenous stochastic overlap oscillations along the DeltaTau/Z scan.

    This is not a DSI template: it has no prescribed frequency, phase, or period.
    It is a correlated random renormalization field applied only to authorized
    global IFS/PCF contact edges.
    """
    phi = float(np.exp(-max(dz, 1e-12) / max(corr_Z, 1e-12)))
    eps_scale = math.sqrt(max(1.0 - phi * phi, 1e-12))
    X = np.zeros((n_steps, n_edges), dtype=float)
    X[0] = rng.normal(size=n_edges)
    for k in range(1, n_steps):
        X[k] = phi * X[k - 1] + eps_scale * rng.normal(size=n_edges)
    # Per-edge centering makes the stochastic factor comparable across contacts.
    X = X - X.mean(axis=0, keepdims=True)
    X = X / np.maximum(X.std(axis=0, keepdims=True), 1e-9)
    return X


def stochastic_contact_weights(structure: Dict[str, Any], Z_grid: np.ndarray, seed: int, sigma: float = 0.28) -> Tuple[List[Edge], np.ndarray]:
    """Positive stochastic conductance weights on C_i^{IFS/PCF} only.

    W_e(Z) = base_e * activation_i(Z) * exp(sigma * X_e(Z) - sigma^2/2)
    where X_e is an endogenous correlated stochastic field.

    There is intentionally no term cos(2pi Z / ln(q) + phi) in the real branch.
    """
    model: IFSModel = structure["model"]
    level: int = structure["level"]
    edges = sorted(structure["edge_to_points"].keys())
    if not edges:
        return edges, np.zeros((len(Z_grid), 0), dtype=float)
    dz = float(np.median(np.diff(Z_grid))) if len(Z_grid) > 1 else 0.1
    rng = np.random.default_rng(seed)
    X = make_ou_process(len(Z_grid), len(edges), rng, dz=dz, corr_Z=0.50 + 0.07 * model.q)
    center = delta_tau_inflection_center(model, level)
    activation = sigmoid(Z_grid, center=center, width=DELTATAU_WIDTH_Z)[:, None]
    base = np.array([edge_base_weight(structure, e) for e in edges], dtype=float)[None, :]
    weights = base * activation * np.exp(sigma * X - 0.5 * sigma * sigma)
    return edges, weights


def laplacian_from_edges(words: List[Word], edges: List[Edge], weights: np.ndarray) -> np.ndarray:
    n = len(words)
    idx = {w: k for k, w in enumerate(sorted(words))}
    L = np.zeros((n, n), dtype=float)
    for edge, wgt in zip(edges, weights):
        a, b = edge
        ia, ib = idx[a], idx[b]
        L[ia, ia] += wgt
        L[ib, ib] += wgt
        L[ia, ib] -= wgt
        L[ib, ia] -= wgt
    return L


def spectrum_from_laplacian(L: np.ndarray) -> np.ndarray:
    if eigvalsh is None:
        raise RuntimeError("scipy.linalg.eigvalsh is required")
    vals = eigvalsh(L)
    vals[vals < 1e-12] = 0.0
    return vals


def heat_trace(vals: np.ndarray, u: float) -> float:
    pos = vals[vals > 1e-12]
    if pos.size == 0:
        return 1.0
    return float(np.mean(np.exp(-u * pos)))


def spectral_summary(vals: np.ndarray) -> Dict[str, float]:
    pos = vals[vals > 1e-12]
    return {
        "n_positive_modes": int(len(pos)),
        "lambda_min_pos": float(np.min(pos)) if len(pos) else np.nan,
        "lambda_median_pos": float(np.median(pos)) if len(pos) else np.nan,
        "lambda_max": float(np.max(vals)) if len(vals) else np.nan,
    }


def robust_poly_residual(x: np.ndarray, y: np.ndarray, degree: int = 3) -> Tuple[np.ndarray, np.ndarray]:
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < degree + 4:
        return y - np.nanmean(y), np.full(degree + 1, np.nan)
    coeff = np.polyfit(x[ok], y[ok], deg=degree)
    trend = np.polyval(coeff, x)
    return y - trend, coeff


def sinusoid_scan(x: np.ndarray, y: np.ndarray, periods: np.ndarray) -> pd.DataFrame:
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    y = y - np.nanmean(y)
    denom = float(np.nansum(y ** 2))
    rows = []
    for T in periods:
        omega = 2 * math.pi / float(T)
        X = np.column_stack([np.sin(omega * x), np.cos(omega * x)])
        try:
            beta, *_ = np.linalg.lstsq(X, y, rcond=None)
            yhat = X @ beta
            power = float(np.sum(yhat ** 2) / denom) if denom > 0 else 0.0
            amp = float(np.sqrt(beta[0] ** 2 + beta[1] ** 2))
        except Exception:
            power = 0.0
            amp = 0.0
        rows.append({"period": float(T), "omega": float(omega), "power": power, "amplitude": amp})
    return pd.DataFrame(rows)


def best_periodicity(Z: np.ndarray, residual: np.ndarray, expected_period: float, seed: int = 0) -> Dict[str, Any]:
    span = float(np.max(Z) - np.min(Z))
    pmin = max(0.08, expected_period * 0.35)
    pmax = min(max(0.3, expected_period * 2.8), span * 0.95)
    if pmax <= pmin:
        pmax = pmin * 2.0
    periods = np.linspace(pmin, pmax, 96)
    scan = sinusoid_scan(Z, residual, periods)
    best = scan.iloc[int(scan["power"].idxmax())].to_dict()
    rng = np.random.default_rng(seed)
    null_best = []
    for _ in range(80):
        yperm = np.array(residual, copy=True)
        rng.shuffle(yperm)
        s = sinusoid_scan(Z, yperm, periods)
        null_best.append(float(s["power"].max()))
    null_best = np.array(null_best)
    best["empirical_p_shuffle"] = float((1 + np.sum(null_best >= best["power"])) / (len(null_best) + 1))
    best["null_power_95"] = float(np.quantile(null_best, 0.95))
    best["power_margin_over_null95"] = float(best["power"] / max(best["null_power_95"], 1e-12))
    best["expected_period_ln_q"] = float(expected_period)
    best["relative_period_error_vs_ln_q"] = float(abs(best["period"] - expected_period) / expected_period)
    return best


def random_edge_set(words: List[Word], n_edges: int, seed: int) -> List[Edge]:
    rng = np.random.default_rng(seed)
    possible = [canonical_edge(a, b) for a, b in itertools.combinations(sorted(words), 2)]
    n_edges = min(n_edges, len(possible))
    if n_edges <= 0:
        return []
    idx = rng.choice(len(possible), size=n_edges, replace=False)
    return [possible[int(i)] for i in idx]


def evaluate_model_seed(structure: Dict[str, Any], seed: int, outdir: str) -> Dict[str, Any]:
    model: IFSModel = structure["model"]
    level: int = structure["level"]
    expected_period = model.expected_Z_period
    # DeltaTau-corrected Z scan: local inflection window only.
    # Reference: ROSG_Test_H1_DeltaTau_Resolution_Coupling uses
    # Zc_i = 1.05 + (i-1)*ln(2), width_Z = 0.34.
    # Here the formula is generalized to ln(q) for multi-IFS PCF models.
    Z_min, Z_max, center = delta_tau_inflection_window(model, level)
    Z_grid = np.linspace(Z_min, Z_max, 72)
    edges, W = stochastic_contact_weights(structure, Z_grid, seed=seed, sigma=0.30)
    assert set(edges).issubset(set(structure["edge_to_points"].keys()))
    assert np.all(W >= 0)

    # Choose a probe diffusion scale from a mid/high-Z representative spectrum.
    mid_idx = int(0.75 * (len(Z_grid) - 1))
    L_mid = laplacian_from_edges(structure["words"], edges, W[mid_idx])
    vals_mid = spectrum_from_laplacian(L_mid)
    pos_mid = vals_mid[vals_mid > 1e-12]
    u_probe = float(1.0 / max(np.median(pos_mid), 1e-12)) if len(pos_mid) else 1.0

    real_rows = []
    spectra_ref = spectral_summary(vals_mid)
    for k, Z in enumerate(Z_grid):
        L = laplacian_from_edges(structure["words"], edges, W[k])
        vals = spectrum_from_laplacian(L)
        p = heat_trace(vals, u_probe)
        real_rows.append({"Z": float(Z), "P_u": p, "logP": float(math.log(max(p, 1e-300))), "mean_weight": float(np.mean(W[k])) if W.shape[1] else 0.0})
    real_df = pd.DataFrame(real_rows)
    resid, coeff = robust_poly_residual(real_df["Z"].to_numpy(), real_df["logP"].to_numpy(), degree=3)
    periodicity = best_periodicity(real_df["Z"].to_numpy(), resid, expected_period, seed=seed + 1000)

    # Random-support control with same node/edge count and the same stochastic mechanism.
    redges = random_edge_set(structure["words"], len(edges), seed=seed + 3000)
    rng = np.random.default_rng(seed + 4000)
    dz = float(np.median(np.diff(Z_grid)))
    RX = make_ou_process(len(Z_grid), len(redges), rng, dz=dz, corr_Z=0.50 + 0.07 * model.q) if redges else np.zeros((len(Z_grid), 0))
    activation = sigmoid(Z_grid, center=center, width=DELTATAU_WIDTH_Z)[:, None]
    Rbase = np.ones((1, len(redges)), dtype=float) * (model.q ** (-0.15 * level))
    RW = Rbase * activation * np.exp(0.30 * RX - 0.5 * 0.30 * 0.30) if redges else np.zeros((len(Z_grid), 0))
    ctrl_logP = []
    for k in range(len(Z_grid)):
        Lr = laplacian_from_edges(structure["words"], redges, RW[k]) if redges else np.zeros((len(structure["words"]), len(structure["words"])))
        rv = spectrum_from_laplacian(Lr)
        ctrl_logP.append(float(math.log(max(heat_trace(rv, u_probe), 1e-300))))
    ctrl_resid, _ = robust_poly_residual(Z_grid, np.array(ctrl_logP), degree=3)
    ctrl_periodicity = best_periodicity(Z_grid, ctrl_resid, expected_period, seed=seed + 5000)

    # Positive detector-only control: separate from the real branch.
    pos_resid = 1.8 * max(float(np.std(resid)), 1e-6) * np.cos(2 * math.pi * Z_grid / expected_period + 0.37)
    pos_periodicity = best_periodicity(Z_grid, pos_resid, expected_period, seed=seed + 6000)

    # Candidate criteria. The real stochastic branch must match ln(q), beat shuffle,
    # and be more specific than the random-support control.
    period_match = bool(periodicity["relative_period_error_vs_ln_q"] <= 0.25)
    shuffle_specific = bool(periodicity["empirical_p_shuffle"] <= 0.05 and periodicity["power_margin_over_null95"] >= 1.20)
    random_support_rejected = bool(ctrl_periodicity["empirical_p_shuffle"] > 0.05 or ctrl_periodicity["power"] < 0.70 * periodicity["power"])
    candidate = bool(period_match and shuffle_specific and random_support_rejected)

    # Long rows for output.
    long = real_df.copy()
    long["model"] = model.name
    long["level"] = level
    long["seed"] = seed
    long["residual_logP"] = resid
    long["control_random_support_residual"] = ctrl_resid
    long["positive_detector_residual"] = pos_resid
    long["u_probe"] = u_probe
    long["expected_period_ln_q"] = expected_period

    return {
        "model": model.name,
        "level": level,
        "seed": seed,
        "u_probe": u_probe,
        "n_cells": len(structure["words"]),
        "n_contact_edges": len(edges),
        "spectral_reference": spectra_ref,
        "periodicity": periodicity,
        "control_random_support_periodicity": ctrl_periodicity,
        "positive_detector_periodicity": pos_periodicity,
        "period_match_pass": period_match,
        "shuffle_specificity_pass": shuffle_specific,
        "random_support_control_rejected": random_support_rejected,
        "stochastic_DSI_candidate": candidate,
        "long_df": long,
    }


def run_audit(outdir: str = "/mnt/data/ROSG_Test6b_IFS_Global_PCF_DeltaTau_Inflection_DSI_Audit_CORRECTED_out") -> Dict[str, Any]:
    os.makedirs(outdir, exist_ok=True)
    models = [
        IFSModel("SG2D_q2_PCF", 2, 2, ((0, 0), (1, 0), (0, 1)), ((0, 0), (1, 0), (0, 1)), True, "Sierpinski-gasket-style 2D p.c.f. contact structure."),
        IFSModel("Vicsek2D_q3_PCF", 2, 3, ((0, 0), (2, 0), (0, 2), (2, 2), (1, 1)), ((0, 0), (1, 0), (0, 1), (1, 1)), True, "Vicsek-cross-style p.c.f. contact structure."),
        IFSModel("Tetra3D_q2_PCF", 3, 2, ((0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)), ((0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)), True, "Tetrahedral p.c.f. control."),
    ]
    # Keep runtime tractable but nontrivial.
    test_level_by_model = {"SG2D_q2_PCF": 4, "Vicsek2D_q3_PCF": 3, "Tetra3D_q2_PCF": 3}
    all_levels_guardrail_by_model = {"SG2D_q2_PCF": [1, 2, 3, 4, 5], "Vicsek2D_q3_PCF": [1, 2, 3], "Tetra3D_q2_PCF": [1, 2, 3]}
    seeds = [11, 23, 37]

    guard_rows, invalid_rows, periodicity_rows, control_rows, positive_rows, spectral_rows = [], [], [], [], [], []
    long_frames = []
    model_seed_results = []

    for model in models:
        for level in all_levels_guardrail_by_model[model.name]:
            st = build_global_contact_structure(model, level)
            guard_rows.append(guardrail_report(st))
            bad = build_invalid_local_fiber_edges(st["words"], set(st["edge_to_points"].keys()))
            bgr = guardrail_report(st, extra_edges=bad)
            bgr.update({"control": "invalid_local_fiber_edges", "invalid_control_applicable": bool(len(bad) > 0), "expected_to_fail_contact_only": bool(len(bad) > 0)})
            invalid_rows.append(bgr)
            invalid_rows.append({"model": model.name, "level": level, "control": "invalid_interlevel_shortcuts", "invalid_control_applicable": True, "same_level_pass": False, "contact_only_pass": False, "global_ifs_pcf_guardrail_pass": False, "expected_to_fail_contact_only": True})

        st_test = build_global_contact_structure(model, test_level_by_model[model.name])
        for seed in seeds:
            res = evaluate_model_seed(st_test, seed=seed + 100 * model.q, outdir=outdir)
            model_seed_results.append(res)
            per = dict(res["periodicity"])
            per.update({"model": model.name, "level": res["level"], "seed": res["seed"], "branch": "real_endogenous_stochastic_overlaps", "stochastic_DSI_candidate": res["stochastic_DSI_candidate"], "period_match_pass": res["period_match_pass"], "shuffle_specificity_pass": res["shuffle_specificity_pass"], "random_support_control_rejected": res["random_support_control_rejected"]})
            periodicity_rows.append(per)
            ctrl = dict(res["control_random_support_periodicity"])
            ctrl.update({"model": model.name, "level": res["level"], "seed": res["seed"], "branch": "CTRL_random_support_same_n_edges"})
            control_rows.append(ctrl)
            pos = dict(res["positive_detector_periodicity"])
            pos.update({"model": model.name, "level": res["level"], "seed": res["seed"], "branch": "POS_detector_only_injected_template"})
            positive_rows.append(pos)
            spec = {"model": model.name, "level": res["level"], "seed": res["seed"], "u_probe": res["u_probe"], "n_cells": res["n_cells"], "n_contact_edges": res["n_contact_edges"]}
            spec.update(res["spectral_reference"])
            spectral_rows.append(spec)
            long_frames.append(res["long_df"])

    guard_df = pd.DataFrame(guard_rows)
    invalid_df = pd.DataFrame(invalid_rows)
    per_df = pd.DataFrame(periodicity_rows)
    ctrl_df = pd.DataFrame(control_rows)
    pos_df = pd.DataFrame(positive_rows)
    spec_df = pd.DataFrame(spectral_rows)
    long_df = pd.concat(long_frames, ignore_index=True) if long_frames else pd.DataFrame()

    all_guardrails_pass = bool(guard_df["global_ifs_pcf_guardrail_pass"].all())
    applicable_invalid = invalid_df[invalid_df.get("invalid_control_applicable", True).astype(bool)]
    invalid_controls_rejected = bool((applicable_invalid["global_ifs_pcf_guardrail_pass"] == False).all()) if not applicable_invalid.empty else False
    stochastic_enabled = True
    external_template_injected = False
    forced_frequency_or_phase_in_real_branch = False
    weights_positive = True
    contact_only_stochastic_support = True
    positive_detector_ok = bool((pos_df["empirical_p_shuffle"] <= 0.05).all()) if not pos_df.empty else False

    # Robust model-level claim: at least 2 of 3 seeds pass for a model; multi-IFS requires at least 2 models.
    candidate_models = []
    if not per_df.empty:
        for model_name, sub in per_df.groupby("model"):
            if int(sub["stochastic_DSI_candidate"].sum()) >= 2:
                candidate_models.append(model_name)
    multi_ifs_stochastic_DSI_candidate = len(candidate_models) >= 2

    if all_guardrails_pass and invalid_controls_rejected and stochastic_enabled and not external_template_injected and multi_ifs_stochastic_DSI_candidate:
        verdict = "Test6b_DeltaTau_inflection_stochastic_overlap_DSI_candidate_on_global_IFS_PCF"
        spontaneous_DSI_claim = True
    elif all_guardrails_pass and invalid_controls_rejected and stochastic_enabled and not external_template_injected:
        verdict = "Test6b_DeltaTau_inflection_global_IFS_PCF_stochastic_overlaps_no_robust_DSI_yet"
        spontaneous_DSI_claim = False
    else:
        verdict = "Test6b_DeltaTau_inflection_inconclusive_guardrail_or_control_failure"
        spontaneous_DSI_claim = False

    report = {
        "status": "completed",
        "test_name": "ROSG_Test6b_IFS_Global_PCF_DeltaTau_Inflection_DSI_Audit_CORRECTED",
        "scope": "Corrected Test6b DeltaTau-inflection: starts from Test5 global IFS/PCF contact set; enables endogenous stochastic overlap oscillations under DeltaTau/Z scan; forbids only an external hand-tuned DSI template in the real branch.",
        "Z_definition": "Z = ln(DeltaTau_sun / t_P)",
        "DeltaTau_inflection_reference": {"Zc_i": "Zc0 + (i-1)*ln(q)", "Zc0": DELTATAU_ZC0, "width_Z": DELTATAU_WIDTH_Z, "window": "[Zc - 2*width_Z, Zc + 2*width_Z]", "source_notebook": "ROSG_Test_H1_DeltaTau_Resolution_Coupling.ipynb", "source_generalization": "ln(2) generalized to ln(q) for multi-IFS PCF models"},
        "restricted_to_DeltaTau_sigmoid_inflection_window": True,
        "models": [asdict(m) for m in models],
        "test_level_by_model": test_level_by_model,
        "seeds": seeds,
        "test5_global_ifs_pcf_guardrails_pass": all_guardrails_pass,
        "invalid_controls_rejected": invalid_controls_rejected,
        "endogenous_stochastic_overlap_oscillations_enabled": stochastic_enabled,
        "oscillation_origin": "contact-edge lognormal OU renormalization field under DeltaTau/Z scan, restricted to C_i^{IFS/PCF}",
        "external_DSI_template_injected_in_real_branch": external_template_injected,
        "forced_frequency_or_phase_in_real_branch": forced_frequency_or_phase_in_real_branch,
        "stochastic_weights_positive": weights_positive,
        "stochastic_weights_contact_only": contact_only_stochastic_support,
        "positive_detector_control_ok": positive_detector_ok,
        "candidate_models_seed_majority": candidate_models,
        "multi_ifs_stochastic_DSI_candidate": multi_ifs_stochastic_DSI_candidate,
        "spontaneous_DSI_claim": spontaneous_DSI_claim,
        "verdict": verdict,
        "important_scope_note": "A false spontaneous_DSI_claim here means the stochastic overlap oscillations, although allowed and active on the Test5 contact set, did not form a robust, specific, multi-IFS log-periodic DSI signature under this implementation. It is not a no-go theorem.",
        "acceptance_criteria": {
            "guardrails": "all Test5 global IFS/PCF contact-only guardrails pass",
            "stochastic_real_branch": "positive endogenous stochastic weights on C_i^{IFS/PCF}; no local fiber support",
            "no_external_template": "no cos(2pi Z/ln(q)+phase) is used in the real branch",
            "period_recovery": "period is recovered by blind residual scan and compared to ln(q)",
            "specificity": "real branch beats shuffle null and rejects random-support control",
            "robustness": "at least 2 seeds per model and at least 2 PCF models for multi-IFS claim"
        }
    }

    outputs = {}
    def save_csv(df, name):
        path = os.path.join(outdir, name)
        df.to_csv(path, index=False)
        outputs[name] = path
        return path

    save_csv(guard_df, "ROSG_Test6b_DELTATAU_INFLECTION_guardrails.csv")
    save_csv(invalid_df, "ROSG_Test6b_DELTATAU_INFLECTION_invalid_controls.csv")
    save_csv(spec_df, "ROSG_Test6b_DELTATAU_INFLECTION_spectral_summary.csv")
    save_csv(per_df, "ROSG_Test6b_DELTATAU_INFLECTION_periodicity_real_stochastic.csv")
    save_csv(ctrl_df, "ROSG_Test6b_DELTATAU_INFLECTION_periodicity_random_support_controls.csv")
    save_csv(pos_df, "ROSG_Test6b_DELTATAU_INFLECTION_positive_detector_controls.csv")
    save_csv(long_df, "ROSG_Test6b_DELTATAU_INFLECTION_Z_scan_observables.csv")

    report_path = os.path.join(outdir, "ROSG_Test6b_DELTATAU_INFLECTION_report.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    outputs["ROSG_Test6b_DELTATAU_INFLECTION_report.json"] = report_path

    # Plots
    if not per_df.empty:
        plt.figure(figsize=(10, 5))
        labels = []
        margins = []
        for _, row in per_df.iterrows():
            labels.append(f"{row['model']}\ns{int(row['seed'])}")
            margins.append(row["power_margin_over_null95"])
        plt.bar(range(len(margins)), margins)
        plt.axhline(1.20, linestyle="--")
        plt.xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=7)
        plt.ylabel("power margin over shuffle null 95%")
        plt.title("Corrected Test6b DeltaTau-inflection: stochastic overlap DSI specificity margin")
        plt.tight_layout()
        p = os.path.join(outdir, "ROSG_Test6b_DELTATAU_INFLECTION_power_margin.png")
        plt.savefig(p, dpi=160)
        plt.close()
        outputs["ROSG_Test6b_DELTATAU_INFLECTION_power_margin.png"] = p

        plt.figure(figsize=(10, 5))
        errs = per_df["relative_period_error_vs_ln_q"].to_numpy()
        plt.bar(range(len(errs)), errs)
        plt.axhline(0.25, linestyle="--")
        plt.xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=7)
        plt.ylabel("relative period error vs ln(q)")
        plt.title("Corrected Test6b DeltaTau-inflection: recovered period vs IFS contraction period")
        plt.tight_layout()
        p = os.path.join(outdir, "ROSG_Test6b_DELTATAU_INFLECTION_period_error.png")
        plt.savefig(p, dpi=160)
        plt.close()
        outputs["ROSG_Test6b_DELTATAU_INFLECTION_period_error.png"] = p

    if not long_df.empty:
        for model_name, subm in long_df.groupby("model"):
            plt.figure(figsize=(9, 5))
            # Plot only first seed for readability.
            s0 = sorted(subm["seed"].unique())[0]
            sub = subm[subm["seed"] == s0]
            plt.plot(sub["Z"], sub["residual_logP"], label="real stochastic overlap residual")
            plt.plot(sub["Z"], sub["control_random_support_residual"], label="random-support control", alpha=0.75)
            plt.plot(sub["Z"], sub["positive_detector_residual"], label="positive detector control", alpha=0.65)
            plt.xlabel("Z = ln(DeltaTau_sun/t_P)")
            plt.ylabel("detrended log P(u_probe; Z)")
            plt.title(f"Corrected Test6b DeltaTau-inflection Z-scan residuals: {model_name}")
            plt.legend(fontsize=8)
            plt.tight_layout()
            p = os.path.join(outdir, f"ROSG_Test6b_DELTATAU_INFLECTION_Z_residual_{model_name}.png")
            plt.savefig(p, dpi=160)
            plt.close()
            outputs[f"ROSG_Test6b_DELTATAU_INFLECTION_Z_residual_{model_name}.png"] = p

    report["outputs"] = outputs
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    return report


def write_readme_and_package(script_path: str, notebook_path: str, outdir: str, package_path: str, readme_path: str, manifest_path: str) -> None:
    readme = """# ROSG Test6b — DeltaTau Inflection Global IFS/PCF Stochastic Overlap DSI Audit

This corrected Test6b starts from the Test5 global IFS/PCF contact-set guardrail and enables endogenous stochastic overlap oscillations under the operational scan `Z = ln(DeltaTau_sun / t_P)`.

## DeltaTau inflection correction

Unlike the broader Test6 scan, Test6b restricts the real audit to the local sigmoid inflection window recovered from `ROSG_Test_H1_DeltaTau_Resolution_Coupling.ipynb`:

- `Zc_i = Zc0 + (i-1)*ln(q)`;
- `Zc0 = 1.05`;
- `width_Z = 0.34`;
- audit window: `[Zc_i - 2*width_Z, Zc_i + 2*width_Z]`.

The original DeltaTau notebook used `ln(2)`; this audit generalizes the contraction spacing to `ln(q)` for multiple PCF IFS models.

## Corrected methodological point

The real branch does **not** suppress oscillations. In ROSG, stochastic overlap oscillations are expected. The audit forbids only an external, hand-tuned DSI template in the real branch, such as a directly imposed `cos(2*pi*Z/ln(q)+phase)` factor.

## Real branch

- support: global same-level contact set `C_i^{IFS/PCF}`;
- DeltaTau locality: local sigmoid inflection window only;
- weights: positive stochastic lognormal/OU contact-edge renormalization field under `Z`;
- no local fiber support;
- no inter-level shortcuts;
- no external DSI frequency/phase template.

## DSI criterion

A candidate requires: contact guardrails pass, stochastic weights remain contact-only and positive, the blind recovered period matches `ln(q)`, the signal beats shuffle nulls, random-support controls are rejected, and at least two PCF models pass by seed-majority.

A negative verdict is not a no-go theorem. It means only that this DeltaTau-inflection implementation did not detect a robust, specific, multi-IFS stochastic-overlap DSI signature.
"""
    Path(readme_path).write_text(readme, encoding="utf-8")
    files = [Path(script_path), Path(notebook_path), Path(readme_path)] + sorted(Path(outdir).glob("*"))
    rows = []
    for f in files:
        if f.exists() and f.is_file():
            h = hashlib.sha256(f.read_bytes()).hexdigest()
            rows.append({"file": f.name, "sha256": h, "bytes": f.stat().st_size})
    pd.DataFrame(rows).to_csv(manifest_path, index=False)
    with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for f in files + [Path(manifest_path)]:
            if f.exists() and f.is_file():
                z.write(f, arcname=f.name)


if __name__ == "__main__":
    result = run_audit()
    print(json.dumps({
        "status": result["status"],
        "verdict": result["verdict"],
        "test5_global_ifs_pcf_guardrails_pass": result["test5_global_ifs_pcf_guardrails_pass"],
        "invalid_controls_rejected": result["invalid_controls_rejected"],
        "endogenous_stochastic_overlap_oscillations_enabled": result["endogenous_stochastic_overlap_oscillations_enabled"],
        "external_DSI_template_injected_in_real_branch": result["external_DSI_template_injected_in_real_branch"],
        "forced_frequency_or_phase_in_real_branch": result["forced_frequency_or_phase_in_real_branch"],
        "positive_detector_control_ok": result["positive_detector_control_ok"],
        "restricted_to_DeltaTau_sigmoid_inflection_window": result["restricted_to_DeltaTau_sigmoid_inflection_window"],
        "candidate_models_seed_majority": result["candidate_models_seed_majority"],
        "multi_ifs_stochastic_DSI_candidate": result["multi_ifs_stochastic_DSI_candidate"],
        "spontaneous_DSI_claim": result["spontaneous_DSI_claim"],
    }, indent=2))


{
  "status": "completed",
  "verdict": "Test6b_DeltaTau_inflection_global_IFS_PCF_stochastic_overlaps_no_robust_DSI_yet",
  "test5_global_ifs_pcf_guardrails_pass": true,
  "invalid_controls_rejected": true,
  "endogenous_stochastic_overlap_oscillations_enabled": true,
  "external_DSI_template_injected_in_real_branch": false,
  "forced_frequency_or_phase_in_real_branch": false,
  "positive_detector_control_ok": true,
  "restricted_to_DeltaTau_sigmoid_inflection_window": true,
  "candidate_models_seed_majority": [],
  "multi_ifs_stochastic_DSI_candidate": false,
  "spontaneous_DSI_claim": false
}


### Notebook Audit Results

The audit completed with the verdict: `Test6b_DeltaTau_inflection_global_IFS_PCF_stochastic_overlaps_no_robust_DSI_yet`. This means that although endogenous stochastic overlap oscillations are enabled, a robust DSI (Dynamic Spectral Instability) signature was not detected in this implementation.

**Key Audit Points:**

*   **Status:** Completed
*   **Verdict:** Test6b_DeltaTau_inflection_global_IFS_PCF_stochastic_overlaps_no_robust_DSI_yet
*   **Global IFS/PCF Guardrails (Test5) Passed:** Yes
*   **Invalid Controls Rejected:** Yes
*   **Endogenous Stochastic Overlap Oscillations Enabled:** Yes
*   **External DSI Template Injected in Real Branch:** No
*   **Forced Frequency or Phase in Real Branch:** No
*   **Positive Detector Control OK:** Yes
*   **Restricted to DeltaTau Sigmoid Inflection Window:** Yes
*   **Stochastic DSI Candidate Models (majority by seed):** []
*   **Multi-IFS Stochastic DSI Candidate:** No
*   **Spontaneous DSI Claim:** No